# STATE SE-600M — Three Embeddings

Open with the **arc-state** kernel and run top-to-bottom.

- Cell 1: native cell embedding
- Cell 2: static gene embedding via `Inference.get_gene_embedding()`
- Cell 3: contextual gene/token representation captured from STATE's decoder output

The contextual output preserves native token ordering. It intentionally does not invent a gene-name mapping.


In [1]:
from pathlib import Path
import sys, time
import numpy as np
import torch
import anndata as ad
from state.emb.inference import Inference

REPO_ROOT = Path(r"E:\state_fork")
CHECKPOINT_PATH = REPO_ROOT/"models"/"SE-600M"/"se600m_epoch16.ckpt"
PROTEIN_EMBEDDINGS_PATH = REPO_ROOT/"models"/"SE-600M"/"protein_embeddings.pt"
INPUT_PATH = REPO_ROOT/"data"/"pbmc3k"/"pbmc3k_state.h5ad"
OUTPUT_DIR = REPO_ROOT/"outputs"/"state_se600m"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Select arc-state kernel with CUDA PyTorch."
assert CHECKPOINT_PATH.exists()
assert PROTEIN_EMBEDDINGS_PATH.exists()
assert INPUT_PATH.exists()

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

adata = ad.read_h5ad(INPUT_PATH)
protein_embeddings = torch.load(PROTEIN_EMBEDDINGS_PATH, map_location="cpu", weights_only=False)

input_genes = set(map(str, adata.var_names))
state_genes = set(map(str, protein_embeddings.keys()))
overlap = input_genes & state_genes

print("Input shape:", adata.shape)
print("STATE protein genes:", len(state_genes))
print("Mapped genes:", len(overlap))
print("Coverage %:", round(100*len(overlap)/len(input_genes), 2))

N_CELL_EMBEDDING_CELLS = 100
N_CONTEXTUAL_CELLS = 1
BATCH_SIZE = 16
TEST_GENES = ["TP53","EGFR","CD3D","NKG7","MS4A1","MALAT1","B2M"]

inferer = Inference(cfg=None, protein_embeds=protein_embeddings)
print("Loading SE-600M...")
inferer.load_model(str(CHECKPOINT_PATH))
inferer.model.eval()
print("d_model:", inferer.model.d_model)
print("dataset dim:", inferer.model.z_dim_ds)
print("RDA dim:", inferer.model.z_dim_rd)
print("Setup complete.")


Python: e:\state_fork\.venv\Scripts\python.exe
PyTorch: 2.6.0+cu124
CUDA: 12.4
GPU: NVIDIA RTX A6000
Input shape: (2700, 32738)
STATE protein genes: 19790
Mapped genes: 17855
Coverage %: 54.54
Loading SE-600M...
d_model: 2048
dataset dim: 10
RDA dim: 1
Setup complete.


In [2]:
# CELL 1 — CELL EMBEDDING
cell_input = OUTPUT_DIR/"pbmc3k_cell_subset.h5ad"
cell_adata = adata[:min(N_CELL_EMBEDDING_CELLS, adata.n_obs)].copy()
cell_adata.write_h5ad(cell_input)

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.perf_counter()

native = inferer.encode_adata(
    input_adata_path=str(cell_input),
    output_adata_path=None,
    batch_size=BATCH_SIZE,
)

torch.cuda.synchronize()
elapsed = time.perf_counter()-t0
peak = torch.cuda.max_memory_allocated()/1024**3
native = np.asarray(native, dtype=np.float32)

ds_dim = int(inferer.model.z_dim_ds)
if ds_dim > 0 and native.shape[1] > ds_dim:
    cell_embeddings = native[:, :-ds_dim]
    dataset_embeddings = native[:, -ds_dim:]
else:
    cell_embeddings, dataset_embeddings = native, None

np.save(OUTPUT_DIR/"cell_embeddings.npy", cell_embeddings)
np.save(OUTPUT_DIR/"cell_embeddings_native_output.npy", native)
if dataset_embeddings is not None:
    np.save(OUTPUT_DIR/"dataset_embeddings.npy", dataset_embeddings)

print("STATE CELL EMBEDDING")
print("Native shape:", native.shape)
print("Cell shape:", cell_embeddings.shape)
print("Dimension:", cell_embeddings.shape[-1])
print("NaN:", np.isnan(cell_embeddings).sum(), "Inf:", np.isinf(cell_embeddings).sum())
print("Time seconds:", round(elapsed,4))
print("Peak GPU VRAM GB:", round(peak,2))


!!! 17855 genes mapped to embedding file (out of 32738)


Encoding: 100%|██████████| 7/7 [00:13<00:00,  1.98s/it]


STATE CELL EMBEDDING
Native shape: (100, 2058)
Cell shape: (100, 2048)
Dimension: 2048
NaN: 0 Inf: 0
Time seconds: 19.9549
Peak GPU VRAM GB: 4.31


In [3]:
# CELL 2 — STATIC / BASE GENE EMBEDDING
genes = [g for g in TEST_GENES if g in protein_embeddings]
print("Genes:", genes)

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.perf_counter()

with torch.no_grad():
    gene_t = inferer.get_gene_embedding(genes)

torch.cuda.synchronize()
elapsed = time.perf_counter()-t0
peak = torch.cuda.max_memory_allocated()/1024**3
gene_embeddings = gene_t.detach().cpu().float().numpy()

np.save(OUTPUT_DIR/"gene_embeddings.npy", gene_embeddings)
np.save(OUTPUT_DIR/"gene_embedding_names.npy", np.asarray(genes, dtype=object))

print("STATE STATIC GENE EMBEDDING")
print("Shape:", gene_embeddings.shape)
print("Dimension:", gene_embeddings.shape[-1])
print("NaN:", np.isnan(gene_embeddings).sum(), "Inf:", np.isinf(gene_embeddings).sum())
print("Time seconds:", round(elapsed,6))
print("Peak GPU VRAM GB:", round(peak,2))


Genes: ['TP53', 'EGFR', 'CD3D', 'NKG7', 'MS4A1', 'B2M']
STATE STATIC GENE EMBEDDING
Shape: (6, 2048)
Dimension: 2048
NaN: 0 Inf: 0
Time seconds: 0.044791
Peak GPU VRAM GB: 1.32


In [4]:
# CELL 3 — CONTEXTUAL GENE/TOKEN EMBEDDING
# Capture STATE's decoder output during its native inference path.
context_input = OUTPUT_DIR/"pbmc3k_context_subset.h5ad"
context_adata = adata[:min(N_CONTEXTUAL_CELLS, adata.n_obs)].copy()
context_adata.write_h5ad(context_input)

captured = []
def capture_decoder(module, inputs, output):
    captured.append(output.detach().cpu().float())

hook = inferer.model.decoder.register_forward_hook(capture_decoder)

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t0 = time.perf_counter()
try:
    _ = inferer.encode_adata(
        input_adata_path=str(context_input),
        output_adata_path=None,
        batch_size=1,
    )
finally:
    hook.remove()

torch.cuda.synchronize()
elapsed = time.perf_counter()-t0
peak = torch.cuda.max_memory_allocated()/1024**3

if not captured:
    raise RuntimeError("No decoder output captured.")

gene_output = torch.cat(captured, dim=0).numpy()

# STATE uses sequence position 0 as CLS.
# With dataset correction, the final position is the dataset token.
if inferer.model.dataset_token is not None:
    contextual_embeddings = gene_output[:, 1:-1, :]
else:
    contextual_embeddings = gene_output[:, 1:, :]

np.save(OUTPUT_DIR/"contextual_gene_embeddings.npy", contextual_embeddings)
np.save(OUTPUT_DIR/"contextual_full_gene_output.npy", gene_output)

print("STATE CONTEXTUAL GENE/TOKEN EMBEDDING")
print("Raw gene_output:", gene_output.shape)
print("Contextual shape:", contextual_embeddings.shape)
print("Dimension:", contextual_embeddings.shape[-1])
print("NaN:", np.isnan(contextual_embeddings).sum(), "Inf:", np.isinf(contextual_embeddings).sum())
print("Time seconds:", round(elapsed,4))
print("Peak GPU VRAM GB:", round(peak,2))
print("NOTE: rows preserve STATE's native sequence-token order. Verify loader token-to-gene ordering before assigning gene names.")


!!! 17855 genes mapped to embedding file (out of 32738)


Encoding: 100%|██████████| 1/1 [00:07<00:00,  7.60s/it]


STATE CONTEXTUAL GENE/TOKEN EMBEDDING
Raw gene_output: (1, 2049, 2048)
Contextual shape: (1, 2047, 2048)
Dimension: 2048
NaN: 0 Inf: 0
Time seconds: 13.4788
Peak GPU VRAM GB: 1.51
NOTE: rows preserve STATE's native sequence-token order. Verify loader token-to-gene ordering before assigning gene names.
